In [ ]:
master="spark://spark-master:7077"

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
             .master(master) \
             .appName('taxi_zone') \
             .getOrCreate()

filename2 = "/data/"
filename = "/data/taxi_zone_lookup.csv"
df = spark.read.option("delimiter", ",").option("header", True).csv(filename)
df.show()



Set up up the Spark master

"local" for local execution
"local[*]" for local execution using all core
"spark://spark-master:7077" to connect to the Spark master running on the docker container
For the first two options you will to set up a Python environment and install pyspark.

tables : |vendor_name|Trip_Pickup_DateTime|Trip_Dropoff_DateTime|Passenger_Count|     Trip_Distance|         Start_Lon|Start_Lat|Rate_Code|store_and_forward|           End_Lon|  End_Lat|Payment_Type|         Fare_Amt|surcharge|mta_tax|Tip_Amt|Tolls_Amt|Total_Amt|

M1.


In [ ]:
from pyspark.sql.functions import col, to_timestamp

# 1. Definir explicitamente quais colunas quer, o tipo correto e a ordem ideal
df_organized = df.select(
    col("Vendor_name").cast("string"),
    # Converte as datas de STRING para TIMESTAMP de forma segura
    to_timestamp(col("Trip_Pickup_DateTime")).alias("Trip_Pickup_DateTime"),
    to_timestamp(col("Trip_Dropoff_DateTime")).alias("Trip_Dropoff_DateTime"),
    col("Passenger_Count").cast("int"),
    col("Trip_Distance").cast("double"),
    col("Fare_Amt").cast("double"),
    col("Tip_Amt").cast("double"),
    col("Tolls_Amt").cast("double"),
    col("Total_Amt").cast("double")
)

# 2. Agora o seu filtro (clean) funcionará perfeitamente e sem erros de tipo
clean = df_organized.where("""
    Fare_Amt > 0
    AND Trip_Dropoff_DateTime IS NOT NULL
    AND Passenger_Count BETWEEN 1 AND 6
    AND Trip_Dropoff_DateTime > Trip_Pickup_DateTime
""")

clean.show()



In [ ]:
M2